# Data Preprocessing for team_2.parquet
This notebook loads the parquet file, explores missing values, applies preprocessing, and saves a cleaned dataset.

In [3]:
# Install pyarrow if required (run once if needed)
# !pip install pyarrow -q

import pandas as pd
import numpy as np
import pyarrow.parquet as pq


In [4]:
# Load parquet file
FILE_PATH = "/Users/aadi/All_Python/data/team_2.parquet"   # Change path if needed

table = pq.read_table(FILE_PATH)
df = table.to_pandas()

print("Shape:", df.shape)
df.head()


Shape: (4445041, 22)


,station_id,state,city,station_name,timestamp,datetime,at_c,rh_percent,ws_m_s,wd_deg,...,sr_w_mt2,bp_mmhg,vws_m_s,pollutant,value,station,year,month,day,hour
0,site_1560,Delhi,Delhi,"Bawana, Delhi - DPCC",2024-01-01T00:00:00.000000+0000,2024-01-01 00:00:00+00:00,11.7,79.5,0.9,351.6,...,5.7,988.1,NaN,co,1.10,"Bawana, Delhi - DPCC",2024,1,1,0
1,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2024-01-01T00:00:00.000000+0000,2024-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,pm25,120.64,"Burari Crossing, Delhi - IMD",2024,1,1,0
2,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2024-01-01T00:00:00.000000+0000,2024-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,pm10,201.09,"Burari Crossing, Delhi - IMD",2024,1,1,0
3,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2024-01-01T00:00:00.000000+0000,2024-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,ozone,19.94,"Burari Crossing, Delhi - IMD",2024,1,1,0
4,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2024-01-01T00:00:00.000000+0000,2024-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,no2,5.14,"Burari Crossing, Delhi - IMD",2024,1,1,0


In [5]:
# Dataset overview
print(df.info())

print("\nMissing Values:")
missing = df.isnull().sum().sort_values(ascending=False)
missing_percent = (missing/len(df))*100

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_percent
})

display(missing_df.head(20))


<class 'pandas.DataFrame'>
RangeIndex: 4445041 entries, 0 to 4445040
Data columns (total 22 columns):
 #   Column        Dtype              
---  ------        -----              
 0   station_id    str                
 1   state         str                
 2   city          str                
 3   station_name  str                
 4   timestamp     str                
 5   datetime      datetime64[us, UTC]
 6   at_c          float64            
 7   rh_percent    float64            
 8   ws_m_s        float64            
 9   wd_deg        float64            
 10  rf_mm         float64            
 11  tot_rf_mm     float64            
 12  sr_w_mt2      float64            
 13  bp_mmhg       float64            
 14  vws_m_s       float64            
 15  pollutant     str                
 16  value         float64            
 17  station       str                
 18  year          int32              
 19  month         int32              
 20  day           int32              
 

,missing_count,missing_percent
vws_m_s,4445041,100.000000
sr_w_mt2,2141530,48.177958
at_c,1913685,43.052134
bp_mmhg,1873922,42.157586
ws_m_s,1734190,39.014038
wd_deg,1449490,32.609148
rh_percent,1326774,29.848409
rf_mm,1272920,28.636856
station_id,0,0.000000
pollutant,0,0.000000


In [6]:
# Drop columns with >95% missing values
threshold = 95

cols_to_drop = missing_percent[missing_percent > threshold].index.tolist()

print("Dropping:", cols_to_drop)

df = df.drop(columns=cols_to_drop)

print("New shape:", df.shape)


Dropping: ['vws_m_s']
New shape: (4445041, 21)


In [7]:
# Convert date/time columns

if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
print(df.dtypes)


station_id                      str
state                           str
city                            str
station_name                    str
timestamp       datetime64[us, UTC]
datetime        datetime64[us, UTC]
at_c                        float64
rh_percent                  float64
ws_m_s                      float64
wd_deg                      float64
rf_mm                       float64
tot_rf_mm                   float64
sr_w_mt2                    float64
bp_mmhg                     float64
pollutant                       str
value                       float64
station                         str
year                          int32
month                         int32
day                           int32
hour                          int32
dtype: object


In [8]:
# Separate column types

numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)


Numeric: ['at_c', 'rh_percent', 'ws_m_s', 'wd_deg', 'rf_mm', 'tot_rf_mm', 'sr_w_mt2', 'bp_mmhg', 'value', 'year', 'month', 'day', 'hour']
Categorical: ['station_id', 'state', 'city', 'station_name', 'pollutant', 'station']


/var/folders/t8/8q56jnnn6wg8qq3zzsxd2t0r0000gn/T/ipykernel_60612/1627905905.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()


In [9]:
# Missing value handling

# Numeric columns:
# Use median imputation grouped by station_id where possible

for col in numeric_cols:
    if col != "station_id":
        if "station_id" in df.columns:
            df[col] = df.groupby("station_id")[col].transform(
                lambda x: x.fillna(x.median())
            )
        df[col] = df[col].fillna(df[col].median())

# Categorical columns:
for col in categorical_cols:
    mode_val = df[col].mode(dropna=True)
    fill_val = mode_val.iloc[0] if len(mode_val) else "Unknown"
    df[col] = df[col].fillna(fill_val)

print(df.isnull().sum().sort_values(ascending=False).head(20))


station_id      0
tot_rf_mm       0
day             0
month           0
year            0
station         0
value           0
pollutant       0
bp_mmhg         0
sr_w_mt2        0
rf_mm           0
state           0
wd_deg          0
ws_m_s          0
rh_percent      0
at_c            0
datetime        0
timestamp       0
station_name    0
city            0
dtype: int64


In [10]:
# Remove duplicate rows

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Removed duplicates:", before-after)


Removed duplicates: 0


In [11]:
# Optional outlier clipping (IQR method)

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr

    df[col] = df[col].clip(lower, upper)

print("Outlier clipping complete")


Outlier clipping complete


In [12]:
# Save cleaned dataset

output_file = "team_2_cleaned.parquet"

df.to_parquet(
    output_file,
    index=False,
    engine="pyarrow"
)

print("Saved:", output_file)

print("Saved:", output_file)
print(df.shape)


Saved: team_2_cleaned.parquet
Saved: team_2_cleaned.parquet
(4445041, 21)
